# Data Fetching & Engineering

In [31]:
import os
import numpy as np
import pandas as pd
from fredapi import Fred
from pathlib import Path

In [2]:
pip install requests

Note: you may need to restart the kernel to use updated packages.


In [3]:
pip install fredapi

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests

In [5]:
os.environ["FRED_API_KEY"] = "656d2723f313fd2fe2e6fceb0b265613"

In [15]:
p0_fred_series = {
    "WEI": "WEI",                  # Weekly Economic Index
    "ICSA": "ICSA",                # Initial Claims
    "T10YIE": "T10YIE",            # 10Y Breakeven Inflation
    "DFII10": "DFII10",            # 10Y Real Yield
    "T10Y3M": "T10Y3M",            # 10Y - 3M yield curve slope
    "SOFR": "SOFR",                # Secured Overnight Financing Rate
    "NFCI": "NFCI",                # Chicago Fed NFCI
    "ANFCI": "ANFCI",              # Adjusted NFCI
    "HY_OAS": "BAMLH0A0HYM2",      # ICE BofA US High Yield OAS
    "VIX": "VIXCLS",               # CBOE VIX
    "EPU": "USEPUINDXD",           # US Daily Economic Policy Uncertainty
}

In [16]:
def get_series_vintage_dates(series_id, api_key,
                             realtime_start=None, realtime_end=None):
    url = "https://api.stlouisfed.org/fred/series/vintagedates"
    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "limit": 10000,
    }
    if realtime_start is not None:
        params["realtime_start"] = realtime_start
    if realtime_end is not None:
        params["realtime_end"] = realtime_end

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    return pd.to_datetime(js["vintage_dates"])

In [17]:
def get_alfred_series_asof(series_id, vintage_date, api_key,
                           observation_start=None, observation_end=None):
    """
    Pull one ALFRED/FRED series as known on `vintage_date`
    using the observations endpoint + vintage_dates parameter.
    """
    url = "https://api.stlouisfed.org/fred/series/observations"

    params = {
        "series_id": series_id,
        "api_key": api_key,
        "file_type": "json",
        "vintage_dates": vintage_date,   # key fix
    }

    if observation_start is not None:
        params["observation_start"] = observation_start
    if observation_end is not None:
        params["observation_end"] = observation_end

    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()

    df = pd.DataFrame(js["observations"])
    df["date"] = pd.to_datetime(df["date"])
    df["value"] = pd.to_numeric(df["value"], errors="coerce")

    s = df.set_index("date")["value"].sort_index()
    s.name = series_id
    return s

In [18]:
def load_p0_fred_asof(vintage_date, api_key,
                      observation_start=None, observation_end=None,
                      series_map=None):
    series_map = p0_fred_series if series_map is None else series_map

    out = {}
    for name, sid in series_map.items():
        out[name] = get_alfred_series_asof(
            series_id=sid,
            vintage_date=vintage_date,
            api_key=api_key,
            observation_start=observation_start,
            observation_end=observation_end,
        )

    df = pd.concat(out, axis=1).sort_index()
    return df

In [19]:
p0_df = load_p0_fred_asof(
    vintage_date="2026-03-07",
    api_key=os.environ['FRED_API_KEY'],
    observation_start="2000-01-01",
)

print(p0_df.tail())

            WEI  ICSA  T10YIE  DFII10  T10Y3M  SOFR  NFCI  ANFCI  HY_OAS  \
date                                                                       
2026-03-02  NaN   NaN    2.29    1.76    0.33  3.71   NaN    NaN    3.03   
2026-03-03  NaN   NaN    2.29    1.77    0.35  3.70   NaN    NaN    3.08   
2026-03-04  NaN   NaN    2.29    1.80    0.38  3.67   NaN    NaN    2.97   
2026-03-05  NaN   NaN    2.31    1.82    0.43  3.66   NaN    NaN    3.00   
2026-03-06  NaN   NaN    2.35     NaN    0.46   NaN   NaN    NaN     NaN   

              VIX     EPU  
date                       
2026-03-02  21.44  441.57  
2026-03-03  23.57  561.10  
2026-03-04  21.15  223.72  
2026-03-05  23.75  552.76  
2026-03-06    NaN     NaN  


In [20]:
def aggregate_vintage_dates(
    api_key,
    series_map=None,
    realtime_start=None,
    realtime_end=None,
):
    """
    Aggregate vintage dates across multiple FRED/ALFRED series.

    Assumes you already have:
        get_series_vintage_dates(series_id, api_key, realtime_start=None, realtime_end=None)

    Returns
    -------
    vintages_by_series : dict[str, pd.DatetimeIndex]
        Vintage dates for each named series.
    all_vintage_dates : pd.DatetimeIndex
        Sorted union of all vintage dates across all series.
    availability_matrix : pd.DataFrame
        Boolean DataFrame indexed by all vintage dates, columns = series names.
        True means that series has a vintage on that date.
    summary : pd.DataFrame
        Simple summary table with first vintage, last vintage, number of vintages.
    """
    series_map = p0_fred_series if series_map is None else series_map

    vintages_by_series = {}
    all_vintage_dates = pd.DatetimeIndex([])

    for name, series_id in series_map.items():
        vdates = get_series_vintage_dates(
            series_id=series_id,
            api_key=api_key,
            realtime_start=realtime_start,
            realtime_end=realtime_end,
        )

        vdates = pd.DatetimeIndex(pd.to_datetime(vdates)).sort_values().unique()
        vintages_by_series[name] = vdates
        all_vintage_dates = all_vintage_dates.union(vdates)

    all_vintage_dates = pd.DatetimeIndex(all_vintage_dates).sort_values()

    availability_matrix = pd.DataFrame(
        False,
        index=all_vintage_dates,
        columns=list(series_map.keys()),
    )

    for name, vdates in vintages_by_series.items():
        if len(vdates) > 0:
            availability_matrix.loc[vdates, name] = True

    summary = pd.DataFrame({
        "series_id": pd.Series(series_map),
        "first_vintage": {
            name: (v[0] if len(v) > 0 else pd.NaT)
            for name, v in vintages_by_series.items()
        },
        "last_vintage": {
            name: (v[-1] if len(v) > 0 else pd.NaT)
            for name, v in vintages_by_series.items()
        },
        "n_vintages": {
            name: len(v)
            for name, v in vintages_by_series.items()
        },
    })

    summary.index.name = "factor"
    summary = summary.sort_values("first_vintage")

    return vintages_by_series, all_vintage_dates, availability_matrix, summary

In [21]:
vintages_by_series, all_vintage_dates, availability_matrix, summary = aggregate_vintage_dates(
    api_key=os.environ['FRED_API_KEY']
)

print(summary)
print(availability_matrix.tail())

           series_id first_vintage last_vintage  n_vintages
factor                                                     
DFII10        DFII10    2005-10-12   2026-03-06        4958
ICSA            ICSA    2009-05-28   2026-03-05         870
VIX           VIXCLS    2010-11-22   2026-03-05        3801
NFCI            NFCI    2011-05-25   2026-03-04         768
ANFCI          ANFCI    2011-05-25   2026-03-04         768
T10YIE        T10YIE    2014-01-27   2026-03-06        2992
T10Y3M        T10Y3M    2014-01-27   2026-03-06        2987
EPU       USEPUINDXD    2014-03-27   2026-03-06        2976
HY_OAS  BAMLH0A0HYM2    2014-04-17   2026-03-05        3106
SOFR            SOFR    2019-03-29   2026-03-06        1731
WEI              WEI    2020-04-16   2026-03-05         397
              WEI   ICSA  T10YIE  DFII10  T10Y3M  SOFR   NFCI  ANFCI  HY_OAS  \
2026-03-02  False  False    True    True    True  True  False  False    True   
2026-03-03  False  False    True    True    True  True  Fals

In [23]:
def build_weekly_model_panel(
    p0_df,
    summary,
    factors=None,
    start_date=None,
    freq="W-FRI",
    min_non_na=1.0,
):
    """
    Provisional model-ready panel from a single as-of snapshot p0_df.

    Notes
    -----
    - Masks out history before each factor's first live vintage.
    - Resamples to weekly snapshots.
    - Keeps only rows with sufficient factor coverage.
    - Good for prototyping / market-based series.
    - Not fully real-time-clean for revised macro series unless p0_df itself
      was rebuilt separately for each as-of date.
    """
    if factors is None:
        factors = [c for c in p0_df.columns if c in summary.index]
    factors = list(factors)

    X = p0_df[factors].copy().sort_index()
    first_live = pd.to_datetime(summary.loc[factors, "first_vintage"])

    # Strict start date
    if start_date is None:
        start_date = first_live.max()
    start_date = pd.Timestamp(start_date)

    X = X.loc[X.index >= start_date]

    # Remove any fake history before factor was actually live
    for f in factors:
        X.loc[X.index < first_live[f], f] = np.nan

    # Weekly snapshot: latest value known by end of week
    Xw = X.resample(freq).last()

    # Coverage filter
    keep = Xw.notna().mean(axis=1) >= min_non_na
    Xw = Xw.loc[keep]

    return Xw

## Long History Core Panel

In [29]:
target_start = pd.Timestamp("2014-05-01")

core_factors = summary.index[
    pd.to_datetime(summary["first_vintage"]) <= target_start
].tolist()

X_core_weekly = build_weekly_model_panel(
    p0_df=p0_df,
    summary=summary,
    factors=core_factors,
    start_date=target_start,
    freq="W-FRI",
    min_non_na=1.0,
)

## Full Panel from 2020

In [27]:
full_factors = [c for c in p0_df.columns if c in summary.index]

full_start = pd.to_datetime(summary.loc[full_factors, "first_vintage"]).max()

X_enriched_weekly = build_weekly_model_panel(
    p0_df=p0_df,
    summary=summary,
    factors=full_factors,
    start_date=full_start,
    freq="W-FRI",
    min_non_na=1.0,
)

In [33]:
path_gdpnow = Path.home()/'Desktop'/'TrackingArchives.csv'
gdpnow = pd.read_csv(path_gdpnow)

In [38]:
# 1) Keep only the two columns we need
gdpnow = (
    gdpnow[["Forecast Date", "GDP Nowcast"]]
    .copy()
    .rename(columns={"GDP Nowcast": "GDPNow"})
)

# 2) Clean types
gdpnow["Forecast Date"] = pd.to_datetime(gdpnow["Forecast Date"])
gdpnow["GDPNow"] = pd.to_numeric(gdpnow["GDPNow"], errors="coerce")

# 3) Build irregular-time series indexed by release date
gdpnow = (
    gdpnow.dropna(subset=["Forecast Date", "GDPNow"])
          .sort_values("Forecast Date")
          .drop_duplicates(subset=["Forecast Date"], keep="last")
          .set_index("Forecast Date")["GDPNow"]
)

# 4) Align to the weekly model dates:
#    for each weekly date, use the latest GDPNow available by that date
gdpnow_weekly = gdpnow.reindex(X_core_weekly.index, method="ffill")
gdpnow_weekly.name = "GDPNow"

# 5) Join into your core weekly feature panel
X_core_weekly = X_core_weekly.join(gdpnow_weekly, how="left")

In [39]:
X_core_weekly.head()

,DFII10,ICSA,VIX,NFCI,ANFCI,T10YIE,T10Y3M,EPU,HY_OAS,GDPNow
date,,,,,,,,,,
2014-05-09,0.44,325000.0,12.92,-0.77575,-0.79137,2.18,2.59,55.25,3.74,3.7
2014-05-16,0.34,303000.0,12.44,-0.77997,-0.79188,2.18,2.49,83.17,3.77,3.2
2014-05-23,0.32,324000.0,11.36,-0.78619,-0.80149,2.22,2.50,80.12,3.78,3.2
2014-05-30,0.26,305000.0,11.40,-0.79203,-0.81517,2.22,2.44,66.44,3.75,2.7
2014-06-06,0.40,312000.0,10.73,-0.79542,-0.82796,2.20,2.56,40.75,3.53,2.7
